In [1]:
# Initial required libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3
import botocore
import s3fs
import smart_open
# from sagemaker import get_execution_role
import seaborn as sns

# Model libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve, brier_score_loss, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef,
    average_precision_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
import joblib
import time, os, threading
from time import perf_counter

### S3 connection

In [ ]:
storage_opts = {
  "key": "AWS_ACCESS_KEY_ID", 
  "secret": "AWS_SECRET_ACCESS_KEY", 
  "token": "AWS_SESSION_TOKEN", 
  "client_kwargs": {"region_name":"us-east-1"}
} # S3 storage options

In [ ]:
df = pd.read_csv("s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv",
                 storage_options=storage_opts) # S3 read with storage options

In [ ]:
# Developing the connection with S3 where we had stored our dataset.

region = storage_opts.get("client_kwargs", {}).get("region_name", "us-east-1")

session = boto3.Session(
    aws_access_key_id=storage_opts["key"],
    aws_secret_access_key=storage_opts["secret"],
    aws_session_token=storage_opts["token"],      # IMPORTANT in Learner Lab
    region_name=region,
) # boto3 session

s3 = session.client("s3") # S3 client

bucket_name = "mlc-project-flight-data-2024"

# use v2 and guard when bucket may be empty
resp = s3.list_objects_v2(Bucket=bucket_name)
contents = resp.get("Contents", [])
print(len(contents), "objects")

# Example: list first 5 keys
for obj in contents[:5]:
    print(obj["Key"])

5 objects
flight_data_2024.csv
flight_data_2024_sample.csv
preprocessed/flight_data_2024_clean.csv
preprocessed/profiles/column_profile.csv
preprocessed/profiles/quick_eda.json


In [ ]:
df.shape # Checking the shape of the dataframe

(7079080, 22)

In [ ]:
df.info() # Checking the info of the dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7079080 entries, 0 to 7079079
Data columns (total 22 columns):
 #   Column               Dtype  
---  ------               -----  
 0   op_unique_carrier    object 
 1   op_carrier_fl_num    float64
 2   origin               object 
 3   origin_city_name     object 
 4   origin_state_nm      object 
 5   dest                 object 
 6   dest_city_name       object 
 7   dest_state_nm        object 
 8   carrier_origin_pair  object 
 9   day_of_month         int64  
 10  day_of_week          int64  
 11  fl_date              object 
 12  crs_dep_time         float64
 13  dep_hour             int64  
 14  crs_arr_time         float64
 15  arr_hour             int64  
 16  crs_elapsed_time     float64
 17  distance             float64
 18  is_weekend           int64  
 19  dep_hour_bin         int64  
 20  arr_delay            float64
 21  is_late              int64  
dtypes: float64(6), int64(7), object(9)
memory usage: 1.2+ GB


In [17]:
# Overview of the dataset
print("--------------------- First 5 rows of the dataset -----------------------")
print(df.head()) # prints first 5 rows of the dataset

--------------------- First 5 rows of the dataset -----------------------
  op_unique_carrier  op_carrier_fl_num origin origin_city_name  \
0                9E             4814.0    JFK     New York, NY   
1                9E             4815.0    MSP  Minneapolis, MN   
2                9E             4817.0    JFK     New York, NY   
3                9E             4817.0    RIC     Richmond, VA   
4                9E             4818.0    DTW      Detroit, MI   

  origin_state_nm dest dest_city_name dest_state_nm carrier_origin_pair  \
0        New York  DTW    Detroit, MI      Michigan              9E_JFK   
1       Minnesota  CLE  Cleveland, OH          Ohio              9E_MSP   
2        New York  RIC   Richmond, VA      Virginia              9E_JFK   
3        Virginia  JFK   New York, NY      New York              9E_RIC   
4        Michigan  MKE  Milwaukee, WI     Wisconsin              9E_DTW   

   day_of_month  ...  crs_dep_time dep_hour  crs_arr_time  arr_hour  \
0      

### Model Specific Training Code

#### Baseline (Dummy) model

In [ ]:
# ---------------- Config ----------------
S3_PATH = "s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv"
LOCAL_FALLBACK = "data/flight_data_2024_clean.csv"   # used if S3 blocked
OUT_DIR = "artifacts/baseline"
os.makedirs(OUT_DIR, exist_ok=True)

# --------------------------- AWS session & creds ----------------------------
# Pull region (and everything else) from the `storage_opts` dict we used for s3fs/pandas.
region = storage_opts.get("client_kwargs", {}).get("region_name", "us-east-1")

# Create a boto3 Session using explicit credentials from storage_opts (works in Learner Lab).
session = boto3.Session(
    aws_access_key_id=storage_opts["key"],
    aws_secret_access_key=storage_opts["secret"],
    aws_session_token=storage_opts["token"],      # IMPORTANT in Learner Lab
    region_name=region)

# Sanity check: make sure credentials actually exist
_creds = session.get_credentials()
if _creds is None:
    raise RuntimeError(
        "No AWS credentials found. Configure env vars or `aws configure`, "
        "or set AWS_PROFILE. Then re-run."
    )
_frozen = _creds.get_frozen_credentials()  # (not used later, shown here for completeness)


# ------------------------- Resource monitoring --------------------
# Try to attach to the current process with psutil, fall back cleanly if missing.
try:
    import psutil
    _PSUTIL_OK = True
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PSUTIL_OK = False
    _PROC = None

# ---------------------------- Feature schema --------------------------------
# Columns used by the model. Must match the columns in the preprocessed CSV.
CATEGORICAL_COLS = ["op_unique_carrier","op_carrier_fl_num","origin","dest","carrier_origin_pair"]
NUMERIC_COLS = ["day_of_month","day_of_week","crs_dep_time","dep_hour","crs_arr_time",
                "arr_hour","crs_elapsed_time","distance","is_weekend","dep_hour_bin"]
TARGET_COL = "is_late"
MAX_TRAINVAL_ROWS = None  # <— use all rows

# # =========================
# 2) Load final table
# =========================
def load_final_table():
    """
    Read the preprocessed table from S3 (or local path), validate required columns,
    and apply light dtypes to reduce memory.
    """
    path = S3_PATH
    try:
        print("Reading from S3:", path)
        return pd.read_csv(path,
                           usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL],
                           storage_options=storage_opts)
    except Exception as e:
        print("S3 read failed -> falling back to local file:", LOCAL_FALLBACK, "\nReason:", e)
        if not os.path.exists(LOCAL_FALLBACK):
            raise FileNotFoundError(f"Local fallback not found: {LOCAL_FALLBACK}")
        return pd.read_csv(LOCAL_FALLBACK,
                           usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL])

def prepare_matrices(df: pd.DataFrame):
    """
    Create 72/8/20 partitions:
      - 80/20 initial split -> X_trainval / X_test
      - 10% of trainval held out for calibration -> X_train / X_calib
    Perform:
      - Ordinal encoding of categoricals (unknown/missing -> -1)
      - Median imputation of numerics
      - Concatenate into dense float32 matrices
    """

    # Separate features/labels
    X = df[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    y = df[TARGET_COL].astype("int8").to_numpy()

    # 80/20 split with stratification to preserve class imbalance
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Optional cap to keep training manageable on small machines
    if MAX_TRAINVAL_ROWS and len(X_trainval) > MAX_TRAINVAL_ROWS:
        idx = np.random.RandomState(42).choice(len(X_trainval), size=MAX_TRAINVAL_ROWS, replace=False)
        X_trainval = X_trainval.iloc[idx].reset_index(drop=True)
        y_trainval = y_trainval[idx]

    # 10% of the remaining trainval becomes the calibration set
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_trainval, y_trainval, test_size=0.1, random_state=42, stratify=y_trainval
    )

    # Encode categoricals, map unseen/missing to -1 so scoring never crashes
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    X_train_cat = enc.fit_transform(X_train[CATEGORICAL_COLS])
    X_calib_cat = enc.transform(X_calib[CATEGORICAL_COLS])
    X_test_cat  = enc.transform(X_test[CATEGORICAL_COLS])

    # Impute numerics with training medians (avoid leakage)
    num_imputer = SimpleImputer(strategy="median")
    X_train_num = num_imputer.fit_transform(X_train[NUMERIC_COLS]).astype(np.float32)
    X_calib_num = num_imputer.transform(X_calib[NUMERIC_COLS]).astype(np.float32)
    X_test_num  = num_imputer.transform(X_test[NUMERIC_COLS]).astype(np.float32)

    # Stack numeric + categorical blocks; use float32 to save memory
    X_train_mat = np.hstack([X_train_num, X_train_cat.astype(np.float32)])
    X_calib_mat = np.hstack([X_calib_num, X_calib_cat.astype(np.float32)])
    X_test_mat  = np.hstack([X_test_num,  X_test_cat.astype(np.float32)])

    # Safety: eliminate any residual NaN/Inf (shouldn't happen but keeps models robust)
    for M in (X_train_mat, X_calib_mat, X_test_mat):
        np.nan_to_num(M, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return X_train_mat, y_train, X_calib_mat, y_calib, X_test_mat, y_test, enc, num_imputer

# --------------------------- Lightweight monitor ----------------------------
class ResourceMonitor:
    """
    Background sampler for avg/peak CPU% and RSS (MB) during model.fit().
    No-op if psutil isn't available.
    """
    def __init__(self, interval=0.2):
        self.interval = interval
        self.samples_cpu = []
        self.samples_mem = []
        self._stop = threading.Event()
        self._t = None

    def _poll(self):
        if _PSUTIL_OK: _PROC.cpu_percent(interval=None)  # prime the meter
        while not self._stop.is_set():
            if _PSUTIL_OK:
                self.samples_cpu.append(_PROC.cpu_percent(interval=None))
                self.samples_mem.append(_PROC.memory_info().rss/(1024**2))  # MB
            time.sleep(self.interval)

    def start(self):
        if _PSUTIL_OK:
            self._t = threading.Thread(target=self._poll, daemon=True)
            self._t.start()

    def stop(self):
        if _PSUTIL_OK and self._t is not None:
            self._stop.set()
            self._t.join()

    def summary(self):
        if not self.samples_cpu:
            return dict(avg_cpu_pct=None, peak_cpu_pct=None, avg_mem_mb=None, peak_mem_mb=None)
        return dict(
            avg_cpu_pct=float(np.mean(self.samples_cpu)),
            peak_cpu_pct=float(np.max(self.samples_cpu)),
            avg_mem_mb=float(np.mean(self.samples_mem)),
            peak_mem_mb=float(np.max(self.samples_mem)),
        )

# ------------------------------- Evaluation ---------------------------------
def evaluate(model, X_test, y_test, desc):
    """
    Report probability-quality metrics (ROC-AUC, PR-AUC, Brier) and
    thresholded classification metrics at a working point (0.25).
    """
    print(f"\n===== {desc} =====")
    y_prob = model.predict_proba(X_test)[:,1]

    # Probabilistic metrics
    auc   = roc_auc_score(y_test, y_prob)
    prauc = average_precision_score(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    print(f"ROC-AUC: {auc:.4f} | PR-AUC: {prauc:.4f} | Brier: {brier:.4f}")

    # Small threshold sweep to show trade-offs
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
    for thr in [0.10,0.20,0.25,0.30,0.40,0.50]:
        if thresholds.size and thresholds.min()<=thr<=thresholds.max():
            idx = np.argmin(np.abs(thresholds - thr))
            print(f"thr={thr:.2f}  P={precisions[idx]:.3f}  R={recalls[idx]:.3f}")

    # Fixed operating point for confusion-matrix based metrics
    best_thr = 0.25
    y_pred = (y_prob >= best_thr).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    tn,fp,fn,tp = cm.ravel()
    acc = accuracy_score(y_test, y_pred)
    bal = balanced_accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan

    print("\nConfusion matrix:\n", cm)
    print(f"Accuracy: {acc:.3f} | Balanced: {bal:.3f} | F1: {f1:.3f} | MCC: {mcc:.3f} | Specificity: {spec:.3f}")
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=3))

# --------------------------------- Main -------------------------------------
def main():
    print("Loading final preprocessed data from S3:", S3_PATH)
    df = load_final_table()

    # Build matrices + preprocessors
    X_train, y_train, X_calib, y_calib, X_test, y_test, enc, num_imp = prepare_matrices(df)

    # Baseline model: always predict the majority class (sanity check)
    model = DummyClassifier(strategy="most_frequent")

    # Fit with resource monitoring
    mon = ResourceMonitor(); mon.start()
    t0 = perf_counter(); model.fit(X_train, y_train); fit_time = perf_counter() - t0
    mon.stop(); stats = mon.summary()

    # Measure prediction latency once
    t1 = perf_counter(); _ = model.predict_proba(X_test); pred_time = perf_counter() - t1

    # Persist a tiny JSON with timing and resource usage
    stats_out = {
        "fit_time_s": round(fit_time,3),
        "pred_time_s": round(pred_time,3),
        **{k:(None if v is None else round(v,2)) for k,v in stats.items()}
    }
    with open(os.path.join(OUT_DIR,"train_resource_summary.json"), "w") as f:
        json.dump(stats_out, f, indent=2)
    print("\nResource summary:", stats_out)

    # Report metrics
    evaluate(model, X_test, y_test, "Dummy (uncalibrated)")

    # Save artifacts (Dummy + preprocessors) to keep the pipeline uniform
    joblib.dump(model, os.path.join(OUT_DIR, "model.joblib"))
    joblib.dump({"encoder": enc, "num_imputer": num_imp}, os.path.join(OUT_DIR, "preprocess.joblib"))
    print(f"Saved artifacts to {OUT_DIR}")

if __name__ == "__main__":
    main()

Loading final preprocessed data from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
Reading from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv

Resource summary: {'fit_time_s': 0.094, 'pred_time_s': 0.006, 'avg_cpu_pct': 71.6, 'peak_cpu_pct': 71.6, 'avg_mem_mb': 935.67, 'peak_mem_mb': 935.67}

===== Dummy (uncalibrated) =====
ROC-AUC: 0.5000 | PR-AUC: 0.2048 | Brier: 0.2048

Confusion matrix:
 [[1125822       0]
 [ 289994       0]]
Accuracy: 0.795 | Balanced: 0.500 | F1: 0.000 | MCC: 0.000 | Specificity: 1.000


/opt/anaconda3/envs/myenv1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/myenv1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/myenv1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Classification report:
               precision    recall  f1-score   support

           0      0.795     1.000     0.886   1125822
           1      0.000     0.000     0.000    289994

    accuracy                          0.795   1415816
   macro avg      0.398     0.500     0.443   1415816
weighted avg      0.632     0.795     0.704   1415816

Saved artifacts to artifacts/baseline


#### Logistic Regression

In [ ]:
# ---------------- Config ----------------
S3_PATH = "s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv"
LOCAL_FALLBACK = "data/flight_data_2024_clean.csv"   # used if S3 read fails
OUT_DIR = "artifacts/logreg"
os.makedirs(OUT_DIR, exist_ok=True)                  # ensure output folder exists

# ---------------- AWS session / credentials ----------------
# Reuse the same creds dict you use for pandas/s3fs (works in Learner Lab too)
region = storage_opts.get("client_kwargs", {}).get("region_name", "us-east-1")
session = boto3.Session(
    aws_access_key_id=storage_opts["key"],
    aws_secret_access_key=storage_opts["secret"],
    aws_session_token=storage_opts["token"],      # required for temporary creds
    region_name=region
)

# Fail early if credentials are missing
_creds = session.get_credentials()
if _creds is None:
    raise RuntimeError(
        "No AWS credentials found. Configure env vars or `aws configure`, "
        "or set AWS_PROFILE. Then re-run."
    )
_frozen = _creds.get_frozen_credentials()  # (not used later; kept for completeness)

# ---------------- Resource monitoring ----------------
# If psutil is available, sample CPU% and RSS while fitting, otherwise no-op.
try:
    import psutil
    _PSUTIL_OK = True
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PSUTIL_OK = False
    _PROC = None

# ---------------- Feature schema ----------------
# Must match the preprocessed CSV
CATEGORICAL_COLS = ["op_unique_carrier","op_carrier_fl_num","origin","dest","carrier_origin_pair"]
NUMERIC_COLS = ["day_of_month","day_of_week","crs_dep_time","dep_hour","crs_arr_time",
                "arr_hour","crs_elapsed_time","distance","is_weekend","dep_hour_bin"]
TARGET_COL = "is_late"
MAX_TRAINVAL_ROWS = None   # cap training rows if needed; None = use all

# ---------------- Data loading ----------------
def load_final_table():
    """
    Read the preprocessed table from S3 (preferred) or local fallback.
    Restrict to the exact columns we need (schema guard).
    """
    try:
        print("Reading from S3:", S3_PATH)
        return pd.read_csv(
            S3_PATH,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL],
            storage_options=storage_opts
        )
    except Exception as e:
        # Any S3/network/permission issue -> fall back to local file
        print("S3 read failed -> fallback local:", LOCAL_FALLBACK, "\nReason:", e)
        if not os.path.exists(LOCAL_FALLBACK):
            raise FileNotFoundError(LOCAL_FALLBACK)
        return pd.read_csv(
            LOCAL_FALLBACK,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
        )

# ---------------- Train/Calib/Test matrices ----------------
def prepare_matrices(df):
    """
    Create 72/8/20 splits:
      - 80/20 -> trainval / test (stratified)
      - 10% of trainval -> calibration set
    Do:
      - Ordinal-encode categoricals (unknown/missing -> -1)
      - Median-impute numerics
      - Concatenate into float32 matrices
    """
    X = df[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    y = df[TARGET_COL].astype("int8").to_numpy()

    # 80/20 initial split (preserve class imbalance)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Optional cap to control compute/memory
    if MAX_TRAINVAL_ROWS and len(X_trainval) > MAX_TRAINVAL_ROWS:
        idx = np.random.RandomState(42).choice(len(X_trainval), size=MAX_TRAINVAL_ROWS, replace=False)
        X_trainval = X_trainval.iloc[idx].reset_index(drop=True)
        y_trainval = y_trainval[idx]

    # 10% of trainval -> calibration set (for probability calibration)
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_trainval, y_trainval, test_size=0.1, random_state=42, stratify=y_trainval
    )

    # Categorical encoding with safe handling of unknown/missing values
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    X_train_cat = enc.fit_transform(X_train[CATEGORICAL_COLS])
    X_calib_cat = enc.transform(X_calib[CATEGORICAL_COLS])
    X_test_cat  = enc.transform(X_test[CATEGORICAL_COLS])

    # Numeric median imputation (computed on training slice to avoid leakage)
    imp = SimpleImputer(strategy="median")
    X_train_num = imp.fit_transform(X_train[NUMERIC_COLS]).astype(np.float32)
    X_calib_num = imp.transform(X_calib[NUMERIC_COLS]).astype(np.float32)
    X_test_num  = imp.transform(X_test[NUMERIC_COLS]).astype(np.float32)

    # Assemble dense matrices (num + cat); keep memory footprint low with float32
    Xt = np.hstack([X_train_num, X_train_cat.astype(np.float32)])
    Xc = np.hstack([X_calib_num, X_calib_cat.astype(np.float32)])
    Xs = np.hstack([X_test_num,  X_test_cat.astype(np.float32)])

    # Final safety: scrub NaN/Inf (rare but prevents surprises at predict time)
    for M in (Xt, Xc, Xs):
        np.nan_to_num(M, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return Xt, y_train, Xc, y_calib, Xs, y_test, enc, imp

# ---------------- Lightweight resource monitor ----------------
class ResourceMonitor:
    """Background sampler for avg/peak CPU% and RSS (MB) during model.fit()."""
    def __init__(self, interval=0.2):
        self.interval = interval
        self.samples_cpu = []
        self.samples_mem = []
        self._stop = threading.Event()
        self._t = None
    def _poll(self):
        if _PSUTIL_OK: _PROC.cpu_percent(interval=None)  # warm up counter
        while not self._stop.is_set():
            if _PSUTIL_OK:
                self.samples_cpu.append(_PROC.cpu_percent(interval=None))
                self.samples_mem.append(_PROC.memory_info().rss/(1024**2))  # MB
            time.sleep(self.interval)
    def start(self): 
        if _PSUTIL_OK:
            self._t = threading.Thread(target=self._poll, daemon=True)
            self._t.start()
    def stop(self):  
        if _PSUTIL_OK and self._t is not None:
            self._stop.set()
            self._t.join()
    def summary(self):
        if not self.samples_cpu:
            return dict(avg_cpu_pct=None, peak_cpu_pct=None, avg_mem_mb=None, peak_mem_mb=None)
        return dict(
            avg_cpu_pct=float(np.mean(self.samples_cpu)),
            peak_cpu_pct=float(np.max(self.samples_cpu)),
            avg_mem_mb=float(np.mean(self.samples_mem)),
            peak_mem_mb=float(np.max(self.samples_mem)),
        )

# ---------------- Evaluation helpers ----------------
def evaluate(model, X_test, y_test, desc):
    """
    Print probability-quality metrics and confusion-matrix metrics at a
    fixed operating point (threshold=0.25), plus a small PR sweep.
    """
    y_prob = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    pr  = average_precision_score(y_test, y_prob)
    b   = brier_score_loss(y_test, y_prob)
    print(f"\n===== {desc} =====\nROC-AUC:{auc:.4f} PR-AUC:{pr:.4f} Brier:{b:.4f}")

    prec, rec, thr = precision_recall_curve(y_test, y_prob)
    for t in [0.10,0.20,0.25,0.30,0.40,0.50]:
        if thr.size and thr.min() <= t <= thr.max():
            i = np.argmin(np.abs(thr - t))
            print(f"thr={t:.2f}  P={prec[i]:.3f}  R={rec[i]:.3f}")

    # Work at threshold 0.25 for downstream interpretability
    t = 0.25
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred); tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y_test, y_pred)
    bal  = balanced_accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    mcc  = matthews_corrcoef(y_test, y_pred)
    spec = tn/(tn+fp) if (tn+fp) > 0 else np.nan
    print("\nConfusion matrix:\n", cm)
    print(f"Accuracy:{acc:.3f} Balanced:{bal:.3f} F1:{f1:.3f} MCC:{mcc:.3f} Specificity:{spec:.3f}")
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=3))

# ---------------- Main ----------------
def main():
    print("Loading final preprocessed data from S3:", S3_PATH)
    df = load_final_table()
    Xt, yt, Xc, yc, Xs, ys, enc, imp = prepare_matrices(df)

    # Core model: Logistic Regression (balanced class weights, lbfgs solver)
    base = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")

    # Fit with resource monitoring & record latency for predict_proba
    mon = ResourceMonitor(); mon.start()
    t0 = perf_counter(); base.fit(Xt, yt); fit_time = perf_counter() - t0
    mon.stop(); stats = mon.summary()
    t1 = perf_counter(); _ = base.predict_proba(Xs); pred_time = perf_counter() - t1

    # Persist resource/timing summary
    eval_stats = {
        "fit_time_s": round(fit_time, 3),
        "pred_time_s": round(pred_time, 3),
        **{k: (None if v is None else round(v, 2)) for k, v in stats.items()}
    }
    with open(os.path.join(OUT_DIR, "train_resource_summary.json"), "w") as f:
        json.dump(eval_stats, f, indent=2)
    print("Resource summary:", eval_stats)

    # Evaluate uncalibrated vs. calibrated (isotonic) models
    evaluate(base, Xs, ys, "LogReg (uncalibrated)")
    calib = CalibratedClassifierCV(estimator=base, method="isotonic", cv="prefit").fit(Xc, yc)
    evaluate(calib, Xs, ys, "LogReg (calibrated)")

    # Save calibrated model and preprocessors for reproducible scoring
    joblib.dump(calib, os.path.join(OUT_DIR, "model.joblib"))
    joblib.dump({"encoder": enc, "num_imputer": imp}, os.path.join(OUT_DIR, "preprocess.joblib"))
    print(f"Saved artifacts to {OUT_DIR}")

if __name__ == "__main__":
    main()


Loading final preprocessed data from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
Reading from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv


/opt/anaconda3/envs/myenv1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Resource summary: {'fit_time_s': 359.38, 'pred_time_s': 0.069, 'avg_cpu_pct': 957.31, 'peak_cpu_pct': 1052.8, 'avg_mem_mb': 920.06, 'peak_mem_mb': 1671.84}

===== LogReg (uncalibrated) =====
ROC-AUC:0.6208 PR-AUC:0.2725 Brier:0.2395
thr=0.20  P=0.205  R=1.000
thr=0.25  P=0.205  R=0.999
thr=0.30  P=0.208  R=0.989
thr=0.40  P=0.235  R=0.862
thr=0.50  P=0.270  R=0.617

Confusion matrix:
 [[   1296 1124526]
 [    259  289735]]
Accuracy:0.206 Balanced:0.500 F1:0.340 MCC:0.003 Specificity:0.001

Classification report:
               precision    recall  f1-score   support

           0      0.833     0.001     0.002   1125822
           1      0.205     0.999     0.340    289994

    accuracy                          0.206   1415816
   macro avg      0.519     0.500     0.171   1415816
weighted avg      0.705     0.206     0.071   1415816


===== LogReg (calibrated) =====
ROC-AUC:0.6207 PR-AUC:0.2708 Brier:0.1582
thr=0.10  P=0.213  R=0.968
thr=0.20  P=0.264  R=0.670
thr=0.25  P=0.277  R=0.52

In [2]:
# Display the resource summary
print("Resource Summary for Logistic Regression model")
with open("artifacts/logreg/train_resource_summary.json","r") as f:
    print(f.read())

Resource Summary for Logistic Regression model
{
  "fit_time_s": 359.38,
  "pred_time_s": 0.069,
  "avg_cpu_pct": 957.31,
  "peak_cpu_pct": 1052.8,
  "avg_mem_mb": 920.06,
  "peak_mem_mb": 1671.84
}


#### Random Forest

In [ ]:
# ---------------- Config ----------------
S3_PATH = "s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv"
LOCAL_FALLBACK = "data/flight_data_2024_clean.csv"  # used only if S3 read fails
OUT_DIR = "artifacts/random_forest"
os.makedirs(OUT_DIR, exist_ok=True)  # ensure output folder exists

# ---------------- AWS session / credentials ----------------
# Reuse the creds dict you already use for pandas/s3fs (Learner Lab compatible).
region = storage_opts.get("client_kwargs", {}).get("region_name", "us-east-1")

session = boto3.Session(
    aws_access_key_id=storage_opts["key"],
    aws_secret_access_key=storage_opts["secret"],
    aws_session_token=storage_opts["token"],      # temporary creds need the token
    region_name=region
)

# Fail early if credentials aren’t actually available
_creds = session.get_credentials()
if _creds is None:
    raise RuntimeError(
        "No AWS credentials found. Configure env vars or `aws configure`, "
        "or set AWS_PROFILE. Then re-run."
    )
_frozen = _creds.get_frozen_credentials()  # (not used later; kept for completeness)

# ---------------- Resource monitoring ----------------
# psutil lets us sample CPU% and RSS during .fit(); if missing, we no-op gracefully.
try:
    import psutil
    _PSUTIL_OK = True
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PSUTIL_OK = False
    _PROC = None

# ---------------- Feature schema ----------------
# Must match the preprocessed CSV columns used to train the model.
CATEGORICAL_COLS = ["op_unique_carrier","op_carrier_fl_num","origin","dest","carrier_origin_pair"]
NUMERIC_COLS = ["day_of_month","day_of_week","crs_dep_time","dep_hour","crs_arr_time",
                "arr_hour","crs_elapsed_time","distance","is_weekend","dep_hour_bin"]
TARGET_COL = "is_late"
MAX_TRAINVAL_ROWS = None  # <- use all training rows (set an int to cap for memory)

# ---------------- Data loading ----------------
def load_final_table():
    """
    Read the preprocessed dataset from S3 (preferred) or a local fallback.
    Restrict to the exact columns required (schema guard).
    """
    try:
        print("Reading from S3:", S3_PATH)
        return pd.read_csv(
            S3_PATH,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL],
            storage_options=storage_opts
        )
    except Exception as e:
        # Network/permission issues -> fall back to local file
        print("S3 read failed -> fallback local:", LOCAL_FALLBACK, "\nReason:", e)
        if not os.path.exists(LOCAL_FALLBACK):
            raise FileNotFoundError(LOCAL_FALLBACK)
        return pd.read_csv(
            LOCAL_FALLBACK,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
        )

# ---------------- Train/Calib/Test matrices ----------------
def prepare_matrices(df):
    """
    Create 72/8/20 splits:
      - 80/20 -> trainval / test (stratified to preserve class ratio)
      - 10% of trainval -> calibration set (for probability calibration)
    Pipeline:
      - Ordinal-encode categoricals (unknown/missing -> -1)
      - Median-impute numerics (fit on training only)
      - Concatenate to float32 design matrices
      - Replace any residual NaN/Inf for safety
    """
    X = df[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    y = df[TARGET_COL].astype("int8").to_numpy()

    # First split: hold out 20% for final testing
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Optional cap of the train+calibration pool (for memory/compute control)
    if MAX_TRAINVAL_ROWS and len(X_trainval) > MAX_TRAINVAL_ROWS:
        idx = np.random.RandomState(42).choice(len(X_trainval), size=MAX_TRAINVAL_ROWS, replace=False)
        X_trainval = X_trainval.iloc[idx].reset_index(drop=True)
        y_trainval = y_trainval[idx]

    # Second split: carve 10% of trainval as a calibration set
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_trainval, y_trainval, test_size=0.1, random_state=42, stratify=y_trainval
    )

    # Ordinal-encode categoricals with robust handling of unknown/missing
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    X_train_cat = enc.fit_transform(X_train[CATEGORICAL_COLS])
    X_calib_cat = enc.transform(X_calib[CATEGORICAL_COLS])
    X_test_cat  = enc.transform(X_test[CATEGORICAL_COLS])

    # Median impute numerics using training distribution; avoid leakage
    imp = SimpleImputer(strategy="median")
    X_train_num = imp.fit_transform(X_train[NUMERIC_COLS]).astype(np.float32)
    X_calib_num = imp.transform(X_calib[NUMERIC_COLS]).astype(np.float32)
    X_test_num  = imp.transform(X_test[NUMERIC_COLS]).astype(np.float32)

    # Assemble dense matrices (numerics + encoded categoricals)
    Xt = np.hstack([X_train_num, X_train_cat.astype(np.float32)])
    Xc = np.hstack([X_calib_num, X_calib_cat.astype(np.float32)])
    Xs = np.hstack([X_test_num,  X_test_cat.astype(np.float32)])

    # Final safety: scrub NaN/Inf to avoid predict-time errors
    for M in (Xt, Xc, Xs):
        np.nan_to_num(M, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return Xt, y_train, Xc, y_calib, Xs, y_test, enc, imp

# ---------------- Lightweight resource monitor ----------------
class ResourceMonitor:
    """Background sampler of avg/peak CPU% and RSS (MB) while model.fit() runs."""
    def __init__(self, interval=0.2):
        self.interval = interval
        self.samples_cpu = []
        self.samples_mem = []
        self._stop = threading.Event()
        self._t = None
    def _poll(self):
        if _PSUTIL_OK: _PROC.cpu_percent(interval=None)  # warm up internal counter
        while not self._stop.is_set():
            if _PSUTIL_OK:
                self.samples_cpu.append(_PROC.cpu_percent(interval=None))
                self.samples_mem.append(_PROC.memory_info().rss/(1024**2))  # MB
            time.sleep(self.interval)
    def start(self):
        if _PSUTIL_OK:
            self._t = threading.Thread(target=self._poll, daemon=True)
            self._t.start()
    def stop(self):
        if _PSUTIL_OK and self._t is not None:
            self._stop.set()
            self._t.join()
    def summary(self):
        if not self.samples_cpu:
            return dict(avg_cpu_pct=None, peak_cpu_pct=None, avg_mem_mb=None, peak_mem_mb=None)
        return dict(
            avg_cpu_pct=float(np.mean(self.samples_cpu)),
            peak_cpu_pct=float(np.max(self.samples_cpu)),
            avg_mem_mb=float(np.mean(self.samples_mem)),
            peak_mem_mb=float(np.max(self.samples_mem)),
        )

# ---------------- Evaluation helpers ----------------
def evaluate(model, X_test, y_test, desc):
    """
    Report probability-quality metrics and confusion-matrix figures at a
    fixed operating point (threshold=0.25), plus a small PR sweep.
    """
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    pr  = average_precision_score(y_test, y_prob)
    b   = brier_score_loss(y_test, y_prob)
    print(f"\n===== {desc} =====\nROC-AUC:{auc:.4f} PR-AUC:{pr:.4f} Brier:{b:.4f}")

    prec, rec, thr = precision_recall_curve(y_test, y_prob)
    for t in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50]:
        if thr.size and thr.min() <= t <= thr.max():
            i = np.argmin(np.abs(thr - t))
            print(f"thr={t:.2f}  P={prec[i]:.3f}  R={rec[i]:.3f}")

    # Work at threshold 0.25 for downstream interpretability
    t = 0.25
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred); tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y_test, y_pred)
    bal  = balanced_accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    mcc  = matthews_corrcoef(y_test, y_pred)
    spec = tn/(tn+fp) if (tn+fp) > 0 else np.nan
    print("\nConfusion matrix:\n", cm)
    print(f"Accuracy:{acc:.3f} Balanced:{bal:.3f} F1:{f1:.3f} MCC:{mcc:.3f} Specificity:{spec:.3f}")
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=3))

# ---------------- Main ----------------
def main():
    print("Loading final preprocessed data from S3:", S3_PATH)
    df = load_final_table()
    Xt, yt, Xc, yc, Xs, ys, enc, imp = prepare_matrices(df)

    # Random Forest: many trees, balanced class weights, full-depth (None) trees
    base = RandomForestClassifier(
        n_estimators=200, max_depth=None, n_jobs=-1,
        class_weight="balanced", random_state=42
    )

    # Fit with resource monitoring; also measure predict_proba latency
    mon = ResourceMonitor(); mon.start()
    t0 = perf_counter(); base.fit(Xt, yt); fit_time = perf_counter() - t0
    mon.stop(); stats = mon.summary()
    t1 = perf_counter(); _ = base.predict_proba(Xs); pred_time = perf_counter() - t1

    # Persist timing + resource stats for the training run
    with open(os.path.join(OUT_DIR, "train_resource_summary.json"), "w") as f:
        json.dump(
            {
                "fit_time_s": round(fit_time, 3),
                "pred_time_s": round(pred_time, 3),
                **{k: (None if v is None else round(v, 2)) for k, v in stats.items()}
            },
            f, indent=2
        )

    # Evaluate raw RF and its calibrated wrapper (isotonic on the 8% calibration split)
    evaluate(base, Xs, ys, "RandomForest (uncalibrated)")
    calib = CalibratedClassifierCV(estimator=base, method="isotonic", cv="prefit").fit(Xc, yc)
    evaluate(calib, Xs, ys, "RandomForest (calibrated)")

    # Save calibrated model + preprocessors for later scoring (consistent pipeline)
    joblib.dump(calib, os.path.join(OUT_DIR, "model.joblib"))
    joblib.dump({"encoder": enc, "num_imputer": imp}, os.path.join(OUT_DIR, "preprocess.joblib"))
    print(f"Saved artifacts to {OUT_DIR}")

if __name__ == "__main__":
    main()

Loading final preprocessed data from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
Reading from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv

===== RandomForest (uncalibrated) =====
ROC-AUC:0.6513 PR-AUC:0.3184 Brier:0.1644
thr=0.10  P=0.256  R=0.784
thr=0.20  P=0.292  R=0.572
thr=0.25  P=0.306  R=0.483
thr=0.30  P=0.322  R=0.406
thr=0.40  P=0.353  R=0.273
thr=0.50  P=0.389  R=0.170

Confusion matrix:
 [[808689 317133]
 [149843 140151]]
Accuracy:0.670 Balanced:0.601 F1:0.375 MCC:0.174 Specificity:0.718

Classification report:
               precision    recall  f1-score   support

           0      0.844     0.718     0.776   1125822
           1      0.306     0.483     0.375    289994

    accuracy                          0.670   1415816
   macro avg      0.575     0.601     0.576   1415816
weighted avg      0.734     0.670     0.694   1415816


===== RandomForest (calibrated) =====
ROC-AUC:0.6513 PR-AUC:0.3164 Brier:0

In [12]:
print("Resource Summary for Random Forest model")
with open("artifacts/random_forest/train_resource_summary.json","r") as f:
    print(f.read())

Resource Summary for Random Forest model
{
  "fit_time_s": 243.82,
  "pred_time_s": 62.905,
  "avg_cpu_pct": 962.11,
  "peak_cpu_pct": 1054.0,
  "avg_mem_mb": 3226.81,
  "peak_mem_mb": 5643.31
}


#### HistGradientBoosting

In [ ]:
# ---------------- Config ----------------
S3_PATH = "s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv"
LOCAL_FALLBACK = "data/flight_data_2024_clean.csv"   # used only if S3 access fails
OUT_DIR = "artifacts/histgb"
os.makedirs(OUT_DIR, exist_ok=True)  # ensure output folder exists

# ---------------- AWS session / credentials ----------------
# Reuse the same creds dict you use for pandas/s3fs reads (Learner Lab compatible).
region = storage_opts.get("client_kwargs", {}).get("region_name", "us-east-1")

session = boto3.Session(
    aws_access_key_id=storage_opts["key"],
    aws_secret_access_key=storage_opts["secret"],
    aws_session_token=storage_opts["token"],      # temporary creds require the session token
    region_name=region
)

# Sanity check: fail fast if creds aren’t actually present
_creds = session.get_credentials()
if _creds is None:
    raise RuntimeError(
        "No AWS credentials found. Configure env vars or `aws configure`, "
        "or set AWS_PROFILE. Then re-run."
    )
_frozen = _creds.get_frozen_credentials()  # not used later; kept for completeness

# ---------------- Resource Monitoring ----------------
# psutil lets us sample CPU% and RSS during .fit(); if missing, we no-op gracefully.
try:
    import psutil
    _PSUTIL_OK = True
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PSUTIL_OK = False
    _PROC = None

# ---------------- Feature schema ----------------
# Must match your preprocessed CSV exactly.
CATEGORICAL_COLS = ["op_unique_carrier","op_carrier_fl_num","origin","dest","carrier_origin_pair"]
NUMERIC_COLS = ["day_of_month","day_of_week","crs_dep_time","dep_hour","crs_arr_time",
                "arr_hour","crs_elapsed_time","distance","is_weekend","dep_hour_bin"]
TARGET_COL = "is_late"
MAX_TRAINVAL_ROWS = None  # <- use all rows (set to an int to cap for memory/compute)

# ---------------- Data loading ----------------
def load_final_table():
    """
    Read the preprocessed dataset from S3 (preferred) or a local fallback,
    selecting only the columns the model expects (schema guard).
    """
    try:
        print("Reading from S3:", S3_PATH)
        return pd.read_csv(
            S3_PATH,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL],
            storage_options=storage_opts
        )
    except Exception as e:
        # Network/permission issues -> fall back to local file
        print("S3 read failed -> fallback local:", LOCAL_FALLBACK, "\nReason:", e)
        if not os.path.exists(LOCAL_FALLBACK):
            raise FileNotFoundError(LOCAL_FALLBACK)
        return pd.read_csv(
            LOCAL_FALLBACK,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
        )

# ---------------- Train/Calib/Test matrices ----------------
def prepare_matrices(df):
    """
    Create 72/8/20 splits:
      - 80/20 -> trainval / test (stratified to preserve class ratio)
      - 10% of trainval -> calibration set (for probability calibration)
    Pipeline:
      - Ordinal-encode categoricals (unknown/missing -> -1)
      - Median-impute numerics (fit on training only)
      - Concatenate to float32 design matrices
      - Replace any residual NaN/Inf for safety
    """
    X = df[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    y = df[TARGET_COL].astype("int8").to_numpy()

    # Hold out 20% for final testing
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Optional cap of train+calibration rows for memory/compute control
    if MAX_TRAINVAL_ROWS and len(X_trainval) > MAX_TRAINVAL_ROWS:
        idx = np.random.RandomState(42).choice(len(X_trainval), size=MAX_TRAINVAL_ROWS, replace=False)
        X_trainval = X_trainval.iloc[idx].reset_index(drop=True)
        y_trainval = y_trainval[idx]

    # From trainval, carve out 10% for calibration
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_trainval, y_trainval, test_size=0.1, random_state=42, stratify=y_trainval
    )

    # Encode categoricals robustly (unknown/missing => -1)
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    X_train_cat = enc.fit_transform(X_train[CATEGORICAL_COLS])
    X_calib_cat = enc.transform(X_calib[CATEGORICAL_COLS])
    X_test_cat  = enc.transform(X_test[CATEGORICAL_COLS])

    # Median-impute numerics using training distribution (avoid leakage)
    imp = SimpleImputer(strategy="median")
    X_train_num = imp.fit_transform(X_train[NUMERIC_COLS]).astype(np.float32)
    X_calib_num = imp.transform(X_calib[NUMERIC_COLS]).astype(np.float32)
    X_test_num  = imp.transform(X_test[NUMERIC_COLS]).astype(np.float32)

    # Assemble dense matrices (numerics + encoded categoricals)
    Xt = np.hstack([X_train_num, X_train_cat.astype(np.float32)])
    Xc = np.hstack([X_calib_num, X_calib_cat.astype(np.float32)])
    Xs = np.hstack([X_test_num,  X_test_cat.astype(np.float32)])

    # Final safety: scrub NaN/Inf to avoid predict-time errors
    for M in (Xt, Xc, Xs):
        np.nan_to_num(M, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return Xt, y_train, Xc, y_calib, Xs, y_test, enc, imp

# ---------------- Lightweight resource monitor ----------------
class ResourceMonitor:
    """Background sampler of avg/peak CPU% and RSS (MB) while model.fit() runs."""
    def __init__(self, interval=0.2):
        self.interval = interval; self.samples_cpu=[]; self.samples_mem=[]
        self._stop = threading.Event(); self._t=None
    def _poll(self):
        if _PSUTIL_OK: _PROC.cpu_percent(interval=None)  # warm up counter
        while not self._stop.is_set():
            if _PSUTIL_OK:
                self.samples_cpu.append(_PROC.cpu_percent(interval=None))
                self.samples_mem.append(_PROC.memory_info().rss/(1024**2))  # MB
            time.sleep(self.interval)
    def start(self):
        if _PSUTIL_OK:
            self._t = threading.Thread(target=self._poll, daemon=True); self._t.start()
    def stop(self):
        if _PSUTIL_OK and self._t is not None:
            self._stop.set(); self._t.join()
    def summary(self):
        if not self.samples_cpu:
            return dict(avg_cpu_pct=None, peak_cpu_pct=None, avg_mem_mb=None, peak_mem_mb=None)
        return dict(
            avg_cpu_pct=float(np.mean(self.samples_cpu)),
            peak_cpu_pct=float(np.max(self.samples_cpu)),
            avg_mem_mb=float(np.mean(self.samples_mem)),
            peak_mem_mb=float(np.max(self.samples_mem)),
        )

# ---------------- Evaluation helpers ----------------
def evaluate(model, X_test, y_test, desc):
    """
    Report probability-quality metrics and confusion-matrix figures at a
    fixed operating point (threshold=0.25), plus a small PR sweep.
    """
    y_prob = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    pr  = average_precision_score(y_test, y_prob)
    b   = brier_score_loss(y_test, y_prob)
    print(f"\n===== {desc} =====\nROC-AUC:{auc:.4f} PR-AUC:{pr:.4f} Brier:{b:.4f}")

    # Quick look at precision/recall around some commonly discussed thresholds
    prec, rec, thr = precision_recall_curve(y_test, y_prob)
    for t in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50]:
        if thr.size and thr.min() <= t <= thr.max():
            i = np.argmin(np.abs(thr - t))
            print(f"thr={t:.2f}  P={prec[i]:.3f}  R={rec[i]:.3f}")

    # Work at threshold 0.25 for downstream interpretability
    t = 0.25
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred); tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y_test, y_pred)
    bal  = balanced_accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    mcc  = matthews_corrcoef(y_test, y_pred)
    spec = tn/(tn+fp) if (tn+fp) > 0 else np.nan
    print("\nConfusion matrix:\n", cm)
    print(f"Accuracy:{acc:.3f} Balanced:{bal:.3f} F1:{f1:.3f} MCC:{mcc:.3f} Specificity:{spec:.3f}")
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=3))

# ---------------- Main ----------------
def main():
    print("Loading final preprocessed data from S3:", S3_PATH)
    df = load_final_table()
    X_train, y_train, X_calib, y_calib, X_test, y_test, enc, imp = prepare_matrices(df)

    # HistGradientBoosting: fast histogram-based tree boosting (scikit-learn)
    base = HistGradientBoostingClassifier(
        loss="log_loss", max_depth=8, learning_rate=0.1, max_iter=200, random_state=42
    )

    # Fit with resource monitoring; also measure predict_proba latency
    mon = ResourceMonitor(); mon.start()
    t0 = perf_counter(); base.fit(X_train, y_train); fit_time = perf_counter() - t0
    mon.stop(); stats = mon.summary()
    t1 = perf_counter(); _ = base.predict_proba(X_test); pred_time = perf_counter() - t1

    # Persist timing + resource stats for the training run
    with open(os.path.join(OUT_DIR, "train_resource_summary.json"), "w") as f:
        json.dump(
            {
                "fit_time_s": round(fit_time, 3),
                "pred_time_s": round(pred_time, 3),
                **{k: (None if v is None else round(v, 2)) for k, v in stats.items()}
            },
            f, indent=2
        )

    # Evaluate raw HGB and its calibrated wrapper (isotonic on the 8% calibration split)
    evaluate(base, X_test, y_test, "HistGB (uncalibrated)")
    calib = CalibratedClassifierCV(estimator=base, method="isotonic", cv="prefit").fit(X_calib, y_calib)
    evaluate(calib, X_test, y_test, "HistGB (calibrated)")

    # Save calibrated model + preprocessors for later scoring (consistent pipeline)
    joblib.dump(calib, os.path.join(OUT_DIR, "model.joblib"))
    joblib.dump({"encoder": enc, "num_imputer": imp}, os.path.join(OUT_DIR, "preprocess.joblib"))
    print(f"Saved artifacts to {OUT_DIR}")

if __name__ == "__main__":
    main()


Loading final preprocessed data from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
Reading from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv

===== HistGB (uncalibrated) =====
ROC-AUC:0.6787 PR-AUC:0.3470 Brier:0.1521
thr=0.10  P=0.222  R=0.964
thr=0.20  P=0.295  R=0.667
thr=0.25  P=0.339  R=0.482
thr=0.30  P=0.385  R=0.309
thr=0.40  P=0.490  R=0.071
thr=0.50  P=0.623  R=0.004

Confusion matrix:
 [[853178 272644]
 [150208 139786]]
Accuracy:0.701 Balanced:0.620 F1:0.398 MCC:0.213 Specificity:0.758

Classification report:
               precision    recall  f1-score   support

           0      0.850     0.758     0.801   1125822
           1      0.339     0.482     0.398    289994

    accuracy                          0.701   1415816
   macro avg      0.595     0.620     0.600   1415816
weighted avg      0.746     0.701     0.719   1415816


===== HistGB (calibrated) =====
ROC-AUC:0.6785 PR-AUC:0.3436 Brier:0.1519
thr=0.

In [ ]:
# Display the resource summary
print("Resource Summary for Histogram Gradient Boosting model")
with open("artifacts/histgb/train_resource_summary.json","r") as f:
    print(f.read())

Resource Summary for Histogram Gradient Boosting model
{
  "fit_time_s": 19.785,
  "pred_time_s": 1.448,
  "avg_cpu_pct": 897.71,
  "peak_cpu_pct": 1057.1,
  "avg_mem_mb": 2478.23,
  "peak_mem_mb": 2546.67
}


#### XG Boost

In [ ]:
# ---------------- Config ----------------
S3_PATH = "s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv"
LOCAL_FALLBACK = "data/flight_data_2024_clean.csv"   # used if S3 access fails
OUT_DIR = "artifacts/xgboost"
os.makedirs(OUT_DIR, exist_ok=True)  # ensure output folder exists

# ---------------- AWS session / credentials ----------------
# Reuse the same creds dict you use for pandas/s3fs reads (Learner Lab compatible).
region = storage_opts.get("client_kwargs", {}).get("region_name", "us-east-1")

session = boto3.Session(
    aws_access_key_id=storage_opts["key"],
    aws_secret_access_key=storage_opts["secret"],
    aws_session_token=storage_opts["token"],      # temporary creds require the session token
    region_name=region
)

# Sanity check: fail fast if creds aren’t actually present
_creds = session.get_credentials()
if _creds is None:
    raise RuntimeError(
        "No AWS credentials found. Configure env vars or `aws configure`, "
        "or set AWS_PROFILE. Then re-run."
    )
_frozen = _creds.get_frozen_credentials()  # not used later; kept for completeness

# ---------------- Resource Monitoring ----------------
# psutil lets us sample CPU% and RSS during .fit(); if missing, we no-op gracefully.
try:
    import psutil
    _PSUTIL_OK = True
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PSUTIL_OK = False
    _PROC = None

# ---------------- Feature schema ----------------
# Must match your preprocessed CSV exactly.
CATEGORICAL_COLS = ["op_unique_carrier","op_carrier_fl_num","origin","dest","carrier_origin_pair"]
NUMERIC_COLS = ["day_of_month","day_of_week","crs_dep_time","dep_hour","crs_arr_time",
                "arr_hour","crs_elapsed_time","distance","is_weekend","dep_hour_bin"]
TARGET_COL = "is_late"
MAX_TRAINVAL_ROWS = None  # <- use all rows (set to an int to cap for memory/compute)

# ---------------- Data loading ----------------
def load_final_table():
    """
    Read the preprocessed dataset from S3 (preferred) or a local fallback,
    selecting only the columns the model expects (schema guard).
    """
    try:
        print("Reading from S3:", S3_PATH)
        return pd.read_csv(
            S3_PATH,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL],
            storage_options=storage_opts
        )
    except Exception as e:
        # Network/permission issues -> fall back to local file
        print("S3 read failed -> fallback local:", LOCAL_FALLBACK, "\nReason:", e)
        if not os.path.exists(LOCAL_FALLBACK):
            raise FileNotFoundError(LOCAL_FALLBACK)
        return pd.read_csv(
            LOCAL_FALLBACK,
            usecols=NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
        )

# ---------------- Train/Calib/Test matrices ----------------
def prepare_matrices(df):
    """
    Create 72/8/20 splits:
      - 80/20 -> trainval / test (stratified to preserve class ratio)
      - 10% of trainval -> calibration set (for probability calibration)
    Pipeline:
      - Ordinal-encode categoricals (unknown/missing -> -1)
      - Median-impute numerics (fit on training only)
      - Concatenate to float32 design matrices
      - Replace any residual NaN/Inf for safety
    """
    X = df[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    y = df[TARGET_COL].astype("int8").to_numpy()

    # Hold out 20% for final testing
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Optional cap of train+calibration rows for memory/compute control
    if MAX_TRAINVAL_ROWS and len(X_trainval) > MAX_TRAINVAL_ROWS:
        idx = np.random.RandomState(42).choice(len(X_trainval), size=MAX_TRAINVAL_ROWS, replace=False)
        X_trainval = X_trainval.iloc[idx].reset_index(drop=True)
        y_trainval = y_trainval[idx]

    # From trainval, carve out 10% for calibration
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_trainval, y_trainval, test_size=0.1, random_state=42, stratify=y_trainval
    )

    # Encode categoricals robustly (unknown/missing => -1)
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    X_train_cat = enc.fit_transform(X_train[CATEGORICAL_COLS])
    X_calib_cat = enc.transform(X_calib[CATEGORICAL_COLS])
    X_test_cat  = enc.transform(X_test[CATEGORICAL_COLS])

    # Median-impute numerics using training distribution (avoid leakage)
    imp = SimpleImputer(strategy="median")
    X_train_num = imp.fit_transform(X_train[NUMERIC_COLS]).astype(np.float32)
    X_calib_num = imp.transform(X_calib[NUMERIC_COLS]).astype(np.float32)
    X_test_num  = imp.transform(X_test[NUMERIC_COLS]).astype(np.float32)

    # Assemble dense matrices (numerics + encoded categoricals)
    Xt = np.hstack([X_train_num, X_train_cat.astype(np.float32)])
    Xc = np.hstack([X_calib_num, X_calib_cat.astype(np.float32)])
    Xs = np.hstack([X_test_num,  X_test_cat.astype(np.float32)])

    # Final safety: scrub NaN/Inf to avoid predict-time errors
    for M in (Xt, Xc, Xs):
        np.nan_to_num(M, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    return Xt, y_train, Xc, y_calib, Xs, y_test, enc, imp

# ---------------- Lightweight resource monitor ----------------
class ResourceMonitor:
    """Background sampler of avg/peak CPU% and RSS (MB) while model.fit() runs."""
    def __init__(self, interval=0.2):
        self.interval = interval; self.samples_cpu=[]; self.samples_mem=[]
        self._stop = threading.Event(); self._t=None
    def _poll(self):
        if _PSUTIL_OK: _PROC.cpu_percent(interval=None)  # warm up counter
        while not self._stop.is_set():
            if _PSUTIL_OK:
                self.samples_cpu.append(_PROC.cpu_percent(interval=None))
                self.samples_mem.append(_PROC.memory_info().rss/(1024**2))  # MB
            time.sleep(self.interval)
    def start(self):
        if _PSUTIL_OK:
            self._t = threading.Thread(target=self._poll, daemon=True); self._t.start()
    def stop(self):
        if _PSUTIL_OK and self._t is not None:
            self._stop.set(); self._t.join()
    def summary(self):
        if not self.samples_cpu:
            return dict(avg_cpu_pct=None, peak_cpu_pct=None, avg_mem_mb=None, peak_mem_mb=None)
        return dict(
            avg_cpu_pct=float(np.mean(self.samples_cpu)),
            peak_cpu_pct=float(np.max(self.samples_cpu)),
            avg_mem_mb=float(np.mean(self.samples_mem)),
            peak_mem_mb=float(np.max(self.samples_mem)),
        )

# ---------------- Evaluation helpers ----------------
def evaluate(model, X_test, y_test, desc):
    """
    Report probability-quality metrics and confusion-matrix figures at a
    fixed operating point (threshold=0.25), plus a small PR sweep.
    """
    y_prob = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    pr  = average_precision_score(y_test, y_prob)
    b   = brier_score_loss(y_test, y_prob)
    print(f"\n===== {desc} =====\nROC-AUC:{auc:.4f} PR-AUC:{pr:.4f} Brier:{b:.4f}")

    # Quick look at precision/recall around some commonly discussed thresholds
    prec, rec, thr = precision_recall_curve(y_test, y_prob)
    for t in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50]:
        if thr.size and thr.min() <= t <= thr.max():
            i = np.argmin(np.abs(thr - t))
            print(f"thr={t:.2f}  P={prec[i]:.3f}  R={rec[i]:.3f}")

    # Work at threshold 0.25 for downstream interpretability
    t = 0.25
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred); tn, fp, fn, tp = cm.ravel()
    acc  = accuracy_score(y_test, y_pred)
    bal  = balanced_accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    mcc  = matthews_corrcoef(y_test, y_pred)
    spec = tn/(tn+fp) if (tn+fp) > 0 else np.nan
    print("\nConfusion matrix:\n", cm)
    print(f"Accuracy:{acc:.3f} Balanced:{bal:.3f} F1:{f1:.3f} MCC:{mcc:.3f} Specificity:{spec:.3f}")
    print("\nClassification report:\n", classification_report(y_test, y_pred, digits=3))

# ---------------- XGBoost model factory ----------------
def train_xgb(X_train, y_train):
    """
    Build an XGBClassifier with imbalance handling via scale_pos_weight.
    We compute the weight as (#neg / #pos) from the training labels.
    """
    num_pos = int((y_train == 1).sum())
    num_neg = int((y_train == 0).sum())
    spw = num_neg / max(num_pos, 1)   # avoid div-by-zero
    print(f"\n[XGBoost] scale_pos_weight ≈ {spw:.2f}")

    model = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist",  # fast histogram algorithm
        eval_metric="logloss", n_jobs=-1, random_state=42,
        objective="binary:logistic", scale_pos_weight=spw          # class imbalance compensation
    )
    return model

# ---------------- Main ----------------
def main():
    print("Loading final preprocessed data from S3:", S3_PATH)
    df = load_final_table()
    X_train, y_train, X_calib, y_calib, X_test, y_test, enc, imp = prepare_matrices(df)

    # Create and fit XGBoost (uncalibrated first)
    base = train_xgb(X_train, y_train)

    # Fit with resource monitoring; also measure predict_proba latency
    mon = ResourceMonitor(); mon.start()
    t0 = perf_counter(); base.fit(X_train, y_train); fit_time = perf_counter() - t0
    mon.stop(); stats = mon.summary()
    t1 = perf_counter(); _ = base.predict_proba(X_test); pred_time = perf_counter() - t1

    # Persist timing + resource stats for the training run
    with open(os.path.join(OUT_DIR, "train_resource_summary.json"), "w") as f:
        json.dump(
            {
                "fit_time_s": round(fit_time, 3),
                "pred_time_s": round(pred_time, 3),
                **{k: (None if v is None else round(v, 2)) for k, v in stats.items()}
            },
            f, indent=2
        )

    # Evaluate raw XGB and its calibrated wrapper (isotonic on the 8% calibration split)
    evaluate(base, X_test, y_test, "XGBoost (uncalibrated)")
    calib = CalibratedClassifierCV(estimator=base, method="isotonic", cv="prefit").fit(X_calib, y_calib)
    evaluate(calib, X_test, y_test, "XGBoost (calibrated)")

    # Save calibrated model + preprocessors for later scoring (consistent pipeline)
    joblib.dump(calib, os.path.join(OUT_DIR, "model.joblib"))
    joblib.dump({"encoder": enc, "num_imputer": imp}, os.path.join(OUT_DIR, "preprocess.joblib"))
    print(f"Saved artifacts to {OUT_DIR}")

if __name__ == "__main__":
    main()

Loading final preprocessed data from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
Reading from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv

[XGBoost] scale_pos_weight ≈ 3.88

===== XGBoost (uncalibrated) =====
ROC-AUC:0.6799 PR-AUC:0.3459 Brier:0.2260
thr=0.10  P=0.205  R=1.000
thr=0.20  P=0.209  R=0.995
thr=0.25  P=0.214  R=0.985
thr=0.30  P=0.224  R=0.961
thr=0.40  P=0.254  R=0.851
thr=0.50  P=0.301  R=0.646

Confusion matrix:
 [[  80086 1045736]
 [   4452  285542]]
Accuracy:0.258 Balanced:0.528 F1:0.352 MCC:0.095 Specificity:0.071

Classification report:
               precision    recall  f1-score   support

           0      0.947     0.071     0.132   1125822
           1      0.214     0.985     0.352    289994

    accuracy                          0.258   1415816
   macro avg      0.581     0.528     0.242   1415816
weighted avg      0.797     0.258     0.177   1415816


===== XGBoost (calibrated) =====
ROC-AUC

In [ ]:
# Display the resource summary
print("Resource Summary for XGBoost model")
with open("artifacts/xgboost/train_resource_summary.json","r") as f:
    print(f.read())

Resource Summary for XGBoost model
{
  "fit_time_s": 9.748,
  "pred_time_s": 0.413,
  "avg_cpu_pct": 1013.06,
  "peak_cpu_pct": 1073.2,
  "avg_mem_mb": 2700.83,
  "peak_mem_mb": 2717.92
}


### Final results and Dashboard ready file for flight delay prediction data for best performing model (XGBoost)

#### XG Boost

In [ ]:
from pathlib import Path
import gc

# Cap native math libs to keep memory/CPU stable on laptop/Jupyter
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "4")

# =================== EDIT THESE PER RUN ===================
MODEL_NAME      = "xgboost"                 # one of: "logreg" | "random_forest" | "histgb" | "xgboost"
MODEL_DIR       = "artifacts/xgboost"       # must contain model.joblib (calibrated) + preprocess.joblib
OUT_CSV         = f"artifacts/{MODEL_NAME}_test20.csv"  # per-model output; will be merged later via hash_key
INCLUDE_CONTEXT = True                      # set True only for the FIRST model you run to include descriptive cols
# ==========================================================

# Data locations
S3_PATH        = "s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv"
LOCAL_FALLBACK = "data/flight_data_2024_clean.csv"

# Stream tuning (chunked read avoids loading the full CSV at once)
CHUNK         = 100_000                     # reduce if you still hit memory pressure
TEST_FRACTION = 0.20                        # deterministic ~20% “test” subset via hashing

# Schema used at training time (must match the trained preprocessors)
CATEGORICAL_COLS = [
    "op_unique_carrier", "op_carrier_fl_num", "origin", "dest", "carrier_origin_pair",
]
NUMERIC_COLS = [
    "day_of_month", "day_of_week", "crs_dep_time", "dep_hour", "crs_arr_time", "arr_hour",
    "crs_elapsed_time", "distance", "is_weekend", "dep_hour_bin",
]
TARGET_COL = "is_late"  # optional: included only if present in source
CONTEXT_COLS = [        # lightweight context columns for Tableau exploration (only in first model’s CSV)
    "op_unique_carrier", "origin", "dest", "dep_hour", "arr_hour",
    "crs_elapsed_time", "distance", "day_of_week", "is_weekend"
]
REQ_FOR_HASH = CATEGORICAL_COLS + NUMERIC_COLS  # columns used to build a stable row hash

def _dtype_map():
    """Downcast numerics to shrink memory footprint while reading."""
    return {
        "op_carrier_fl_num": "float32",
        "day_of_month": "int16",
        "day_of_week": "int16",
        "crs_dep_time": "float32",
        "dep_hour": "int8",
        "crs_arr_time": "float32",
        "arr_hour": "int8",
        "crs_elapsed_time": "float32",
        "distance": "float32",
        "is_weekend": "int8",
        "dep_hour_bin": "int8",
        "is_late": "int8",
    }

# ---- Build storage_options for pandas s3:// reads (Learner Lab/env creds)
def build_storage_opts():
    session = boto3.Session()  # pulls AWS_* env vars or ~/.aws automatically
    creds = session.get_credentials()
    if creds is None:
        raise RuntimeError("No AWS credentials. Export AWS_* or run `aws configure`.")
    frozen = creds.get_frozen_credentials()
    region = session.region_name or "us-east-1"
    return {
        "key": frozen.access_key,
        "secret": frozen.secret_key,
        "token": frozen.token,
        "client_kwargs": {"region_name": region},
    }

def _needs_s3(p: str) -> bool:
    """Cheap check: is this an s3:// URI?"""
    return str(p).startswith("s3://")

def _read_header(path: str):
    """Probe the header only; also verifies S3 accessibility early."""
    kw = dict(nrows=0)
    if _needs_s3(path): kw["storage_options"] = storage_opts
    return pd.read_csv(path, **kw).columns.tolist()

def _read_chunks(path: str, chunksize: int):
    """
    Stream the file in chunks with the exact columns we need.
    Using engine='c' and a dtype map keeps it fast and memory-light.
    """
    usecols = list(set(CATEGORICAL_COLS + NUMERIC_COLS + [TARGET_COL]))
    kw = dict(chunksize=chunksize, usecols=usecols, dtype=_dtype_map(), engine="c")
    if _needs_s3(path): kw["storage_options"] = storage_opts
    return pd.read_csv(path, **kw)

def _write_chunk(df: pd.DataFrame, out_csv: str, first: bool):
    """Append-friendly CSV writer (writes header only on the first chunk)."""
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False, mode=("w" if first else "a"), header=first)

# ---- Deterministic 20% mask using a stable hash over key columns
def pick_test_mask(df: pd.DataFrame, frac: float = TEST_FRACTION) -> np.ndarray:
    """
    Build a stable 20% subset without loading the full dataset:
    - Hash concatenation of selected columns per row
    - Keep rows where (hash % 10000) < frac*10000
    Results are deterministic across runs/files with same content.
    """
    key_df = df[[c for c in REQ_FOR_HASH if c in df.columns]].copy()
    for c in key_df.columns:
        key_df[c] = key_df[c].astype("string")
    h = pd.util.hash_pandas_object(key_df, index=False).to_numpy(dtype="uint64")
    thresh = int(frac * 10000)
    return (h % 10000) < thresh, h  # return mask + raw hash to build hash_key

# ---- Load artifacts for THIS model (calibrated classifier + preprocessors)
def load_artifacts(model_dir: str):
    model_path = Path(model_dir, "model.joblib")
    prep_path  = Path(model_dir, "preprocess.joblib")
    if not model_path.exists() or not prep_path.exists():
        raise FileNotFoundError(f"Missing artifacts in {model_dir}: model.joblib / preprocess.joblib")
    model = joblib.load(model_path)     # calibrated model (CalibratedClassifierCV)
    pre   = joblib.load(prep_path)      # {"encoder": OrdinalEncoder, "num_imputer": SimpleImputer}
    return model, pre["encoder"], pre["num_imputer"]

model, ENC, IMP = load_artifacts(MODEL_DIR)

# ---- Transform one chunk into the design matrix expected by the model
def transform_block(block: pd.DataFrame) -> np.ndarray:
    """
    Apply the SAME preprocessing used in training:
      - OrdinalEncoder for categoricals (already fitted)
      - Median imputer for numerics (already fitted)
      - Concatenate -> float32 array, sanitize NaN/Inf
    """
    X_cat = ENC.transform(block[CATEGORICAL_COLS])
    if not isinstance(X_cat, np.ndarray): X_cat = np.asarray(X_cat)
    X_cat = X_cat.astype("float32", copy=False)

    X_num = IMP.transform(block[NUMERIC_COLS])
    if not isinstance(X_num, np.ndarray): X_num = np.asarray(X_num)
    X_num = X_num.astype("float32", copy=False)

    X = np.concatenate((X_num, X_cat), axis=1)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X

def main():
    # Choose data source (prefer S3; fall back to local if needed)
    in_path = S3_PATH
    try:
        _ = _read_header(S3_PATH)
        print(f"Reading from S3: {S3_PATH}")
    except Exception as e:
        print(f"⚠️  Could not read S3 ({e}); switching to local: {LOCAL_FALLBACK}")
        in_path = LOCAL_FALLBACK
        if not Path(in_path).exists():
            raise FileNotFoundError(f"Local fallback not found: {in_path}")

    # Fresh output file
    if Path(OUT_CSV).exists(): Path(OUT_CSV).unlink()
    first = True
    kept = scanned = 0

    # Stream the source CSV; select deterministic ≈20% rows; score; write
    for chunk in _read_chunks(in_path, CHUNK):
        scanned += len(chunk)

        # Build the test subset mask BEFORE any heavy transforms
        m, h = pick_test_mask(chunk)
        if not m.any():
            continue

        sub = chunk.loc[m].copy()

        # Build a stable key that will be identical across models
        key_df = sub[[c for c in REQ_FOR_HASH if c in sub.columns]].astype("string")
        hash_key = pd.util.hash_pandas_object(key_df, index=False).astype("uint64")
        sub_out = pd.DataFrame({"hash_key": hash_key})

        # Add context columns only once (first model) to keep other CSVs lean
        if INCLUDE_CONTEXT:
            for c in CONTEXT_COLS:
                if c in sub.columns:
                    sub_out[c] = sub[c]
            if TARGET_COL in sub.columns:
                sub_out[TARGET_COL] = sub[TARGET_COL]

        # Predict calibrated probabilities (model is already CalibratedClassifierCV)
        X = transform_block(sub)
        sub_out[f"pred_{MODEL_NAME}"] = model.predict_proba(X)[:, 1].astype("float32", copy=False)

        # Append to CSV and free memory promptly
        _write_chunk(sub_out, OUT_CSV, first)
        first = False
        kept += len(sub_out)
        print(f"  scanned {scanned:,} | wrote {kept:,} rows to {OUT_CSV}", flush=True)

        del sub, sub_out, X, m, h, chunk
        gc.collect()

    print(f"✅ Done: {OUT_CSV}")

if __name__ == "__main__":
    main()

Reading from S3: s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
  scanned 100,000 | wrote 20,050 rows to artifacts/xgboost_test20.csv
  scanned 200,000 | wrote 40,089 rows to artifacts/xgboost_test20.csv
  scanned 300,000 | wrote 60,077 rows to artifacts/xgboost_test20.csv
  scanned 400,000 | wrote 80,244 rows to artifacts/xgboost_test20.csv
  scanned 500,000 | wrote 100,285 rows to artifacts/xgboost_test20.csv
  scanned 600,000 | wrote 120,365 rows to artifacts/xgboost_test20.csv
  scanned 700,000 | wrote 140,331 rows to artifacts/xgboost_test20.csv
  scanned 800,000 | wrote 160,429 rows to artifacts/xgboost_test20.csv
  scanned 900,000 | wrote 180,433 rows to artifacts/xgboost_test20.csv
  scanned 1,000,000 | wrote 200,312 rows to artifacts/xgboost_test20.csv
  scanned 1,100,000 | wrote 220,257 rows to artifacts/xgboost_test20.csv
  scanned 1,200,000 | wrote 240,192 rows to artifacts/xgboost_test20.csv
  scanned 1,300,000 | wrote 260,279 rows to artifacts/x